## Formate prompts

### Read dataset and add some features

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import skew, kurtosis

df = pd.read_csv('../data/all.csv')

# Replace NaNs with new class
df['term_id'] = df['term_id'].fillna(-1)  # or any unique sentinel value

df['gender'] = df.gender.map({0: "женщина", 1: "мужчина"})

# Ensure datetime is parsed
df['tr_datetime'] = pd.to_datetime(df['tr_datetime'], errors='coerce')
assert np.issubdtype(df['tr_datetime'].dtype, np.datetime64), "tr_datetime must be parsed"

# Sort by datetime
df.sort_values(['customer_id', 'tr_datetime'], ascending=[True, True], inplace=True)

df

In [ ]:

# ─────────────────────────────────────
# Feature Engineering Per Customer
# ─────────────────────────────────────

# 1. Basic transaction stats + counts of unique categorical features
tx_stats = df.groupby('customer_id').agg({
    'amount': ['count', 'sum', 'mean', 'std', 'min', 'max'],
    'term_id': pd.Series.nunique,
    'mcc_code': pd.Series.nunique,
    'tr_type': pd.Series.nunique
})

# Flatten multiindex columns
tx_stats.columns = [f'base_{k}_{stat}' for k, stat in tx_stats.columns]
tx_stats.reset_index(inplace=True)

# 2. Positive vs negative transaction breakdown
df['is_positive'] = (df['amount'] > 0).astype(int)
df['is_negative'] = (df['amount'] < 0).astype(int)

pos_neg_stats = df.groupby('customer_id').agg({
    'is_positive': 'mean',
    'is_negative': 'mean'
}).rename(columns={
    'is_positive': 'share_positive_txn',
    'is_negative': 'share_negative_txn'
}).reset_index()

# 3. Enhanced amount aggregations
df['amount_positive'] = df['amount'].where(df['amount'] > 0, 0)
df['amount_negative'] = df['amount'].where(df['amount'] < 0, 0)

agg_funcs = {
    'amount': ['median',
               lambda x: np.percentile(x, 25),
               lambda x: np.percentile(x, 75),
               skew,
               kurtosis
               ],
    'amount_positive': ['sum', 'count'],
    'amount_negative': ['sum', 'count']
}

amount_stats = df.groupby('customer_id').agg(agg_funcs)

# Rename columns for clarity
amount_stats.columns = [
    'amount_median',
    'amount_pct25',
    'amount_pct75',
    'amount_skew',
    'amount_kurtosis',
    'amount_positive_sum',
    'amount_positive_count',
    'amount_negative_sum',
    'amount_negative_count'
]

# Calculate ratios and shares

amount_stats['amount_pos_count_share'] = (
    amount_stats['amount_positive_count'] / (amount_stats['amount_positive_count'] + amount_stats['amount_negative_count'] + 1e-9)
)

# Handle inf and NaN
amount_stats.replace([np.inf, -np.inf], 0, inplace=True)
amount_stats.fillna(0, inplace=True)
amount_stats.reset_index(inplace=True)

# 3. Temporal patterns
df['hour'] = df['tr_datetime'].dt.hour
df['minute'] = df['tr_datetime'].dt.minute
df['weekday'] = df['tr_datetime'].dt.weekday
df['day'] = (df['tr_datetime'] - pd.to_datetime("2000-01-01")).dt.days
df['is_weekend'] = df['weekday'].isin([5, 6]).astype(int)
df['days_since_last_txn'] = df.groupby('customer_id')['day'].transform(lambda x: x.max() - x)

weekday_map = {0: 'Понедельник', 1: 'Вторник', 2: 'Среда', 3: 'Четверг', 4: 'Пятница', 5: 'Суббота', 6: 'Воскресенье'}

df['weekday_name'] = df['weekday'].map(weekday_map)

temporal_stats = df.groupby('customer_id').agg({
    'minute': ['mean', 'std', 'min', 'max'],
    'weekday': ['mean', 'std', pd.Series.nunique],
    'is_weekend': 'mean',
    'day': ['min', 'max', pd.Series.nunique],
    'days_since_last_txn': 'mean'
})

temporal_stats.columns = [f'temp_{k}_{stat}' for k, stat in temporal_stats.columns]
temporal_stats.reset_index(inplace=True)

weekday_freq = (
    df.groupby(['customer_id', 'weekday_name'])
    .size()
    .reset_index(name='count')
    .groupby('customer_id')
    .apply(lambda x: dict(sorted(zip(x['weekday_name'], x['count']), key=lambda kv: kv[1], reverse=True)))
    .reset_index(name='weekday_freq_dict')
)

hour_freq = (
    df.groupby(['customer_id', 'hour'])
    .size()
    .reset_index(name='count')
    .groupby('customer_id')
    .apply(lambda x: dict(sorted(zip(x['hour'], x['count']), key=lambda kv: kv[1], reverse=True)))
    .reset_index(name='hour_freq_dict')
)

temporal_stats = temporal_stats \
    .merge(weekday_freq, on='customer_id', how='left') \
    .merge(hour_freq, on='customer_id', how='left') 

# 4. MCC code frequency as dict
mcc_freq = (
    df.groupby(['customer_id', 'mcc_code_desc'])
    .size()
    .reset_index(name='count')
    .groupby('customer_id')
    .apply(lambda x: dict(sorted(zip(x['mcc_code_desc'], x['count']), key=lambda kv: kv[1], reverse=True)))
    .reset_index(name='mcc_freq_dict')
)

# 5. Transaction type frequency as dict
trtype_freq = (
    df.groupby(['customer_id', 'tr_type_desc'])
    .size()
    .reset_index(name='count')
    .groupby('customer_id')
    .apply(lambda x: dict(sorted(zip(x['tr_type_desc'], x['count']), key=lambda kv: kv[1], reverse=True)))
    .reset_index(name='trtype_freq_dict')
)

# 6. Transaction density
txn_density = df.groupby('customer_id')['day'].agg(['count', 'nunique'])
txn_density['txn_per_day'] = txn_density['count'] / txn_density['nunique']
txn_density = txn_density[['txn_per_day']].reset_index()


# ─────────────────────────────────────
# Merge all features
# ─────────────────────────────────────
features = tx_stats \
    .merge(pos_neg_stats, on='customer_id', how='left') \
    .merge(amount_stats, on='customer_id', how='left') \
    .merge(temporal_stats, on='customer_id', how='left') \
    .merge(txn_density, on='customer_id', how='left') \
    .merge(mcc_freq, on='customer_id', how='left') \
    .merge(trtype_freq, on='customer_id', how='left') 

# Add target
target = df[['customer_id', 'gender']].drop_duplicates()
features = features.merge(target, on='customer_id', how='left')

# Final data
X = features.drop(columns=['customer_id', 'gender'])
y = features['gender']

In [ ]:
features

### Add readable stats per user

In [ ]:
readable_features = features.drop(columns=["gender", "customer_id"]).rename(columns={
    "base_amount_count": "Количество всех транзакций клиента",
    "base_amount_sum": "Общая сумма всех транзакций клиента",
    "base_amount_mean": "Средняя сумма одной транзакции",
    "base_amount_std": "Стандартное отклонение суммы транзакций",
    "base_amount_min": "Минимальная сумма транзакции",
    "base_amount_max": "Максимальная сумма транзакции",
    "base_term_id_nunique": "Количество уникальных терминалов, где проводились транзакции",
    "base_mcc_code_nunique": "Количество уникальных MCC кодов (категорий трат)",
    "base_tr_type_nunique": "Количество уникальных типов транзакций",
    "share_positive_txn": "Доля транзакций с положительной суммой (поступления)",
    "share_negative_txn": "Доля транзакций с отрицательной суммой (расходы)",
    "amount_median": "Медианная сумма транзакции",
    "amount_pct25": "25-й перцентиль суммы транзакции",
    "amount_pct75": "75-й перцентиль суммы транзакции",
    "amount_skew": "Асимметрия распределения сумм транзакций",
    "amount_kurtosis": "Эксцесс (острота) распределения сумм транзакций",
    "amount_positive_sum": "Общая сумма всех положительных транзакций",
    "amount_positive_count": "Количество положительных транзакций",
    "amount_negative_sum": "Общая сумма всех отрицательных транзакций",
    "amount_negative_count": "Количество отрицательных транзакций",
    "amount_pos_count_share": "Доля положительных транзакций по количеству",
    "temp_minute_mean": "Среднее значение минут проведения транзакций (например, по времени суток)",
    "temp_minute_std": "Стандартное отклонение минут проведения транзакций",
    "temp_minute_min": "Минимальное значение минут проведения транзакций",
    "temp_minute_max": "Максимальное значение минут проведения транзакций",
    "temp_weekday_mean": "Средний день недели транзакций (например, ближе к будням или выходным)",
    "temp_weekday_std": "Стандартное отклонение по дням недели транзакций",
    "temp_weekday_nunique": "Количество уникальных дней недели, в которые были транзакции",
    "temp_is_weekend_mean": "Средняя доля транзакций, которые происходят в выходные",
    "temp_day_min": "Минимальный календарный день транзакций",
    "temp_day_max": "Максимальный календарный день транзакций",
    "temp_day_nunique": "Количество уникальных календарных дней с транзакциями",
    "temp_days_since_last_txn_mean": "Среднее количество дней между транзакциями",
    "txn_per_day": "Среднее количество транзакций в день",
    "mcc_freq_dict": "Словарь частот по категориям MCC кодов",
    "trtype_freq_dict": "Словарь частот по типам транзакций",
})

# stats_descriptions_list = readable_features.apply(lambda x: ", ".join([f"{k} = {v}" for k, v in x.items()]), axis=1).to_list()
stats_descriptions_list = readable_features.to_dict(orient='records')

print(stats_descriptions_list[0])

### Add readable table

In [ ]:
from tqdm import tqdm
tqdm.pandas()

column2readable_name = {
    "tr_datetime": "дата и время проведения транзакции",
    "amount": "сумма транзакции",
    "tr_type_desc": "описание типа транзакции (например, покупка, возврат и т.д.)",
    "mcc_code_desc": "описание MCC-кода, определяющего категорию торговой точки",
    "hour": "час проведения транзакции",
    "minute": "минута проведения транзакции",
    "weekday": "день недели, когда произошла транзакция",
    "day": "день месяца транзакции",
    "is_weekend": "флаг, указывающий, была ли транзакция в выходной день",
    "days_since_last_txn": "количество дней, прошедших с последней транзакции этого клиента"
}

df_readable_columns = df.drop(columns=["term_id", "gender", "mcc_code", "tr_type", "amount_positive", "amount_negative"]).rename(columns=column2readable_name)
markdown_tables_last_100_list = df_readable_columns.groupby("customer_id").progress_apply(lambda x: x[-50:].to_markdown()).to_list()
markdown_tables_first_100_list = df_readable_columns.groupby("customer_id").progress_apply(lambda x: x[:50].to_markdown()).to_list()
print(markdown_tables_first_100_list[0])

### Construct prompts

In [ ]:
def generate_reasoning_prompt(md_table_first: str, md_table_last: str,  aggregates: dict, true_label: str) -> dict:
    """
    Генерирует промпт для reasoning модели с разделением на системный и пользовательский контекст.

    Args:
        md_table (str): Таблица транзакций клиента в формате Markdown.
        aggregates (dict): Агрегированные показатели по всем транзакциям клиента.
        true_label (str): Истинная метка пола ("Мужчина" или "Женщина").

    Returns:
        dict: Словарь с ключами "system" и "user".
    """

    # Форматируем агрегаты в строку
    agg_str = ", ".join([f"{k} = {v}" for k, v in aggregates.items()])

    system_prompt = "Ты эксперт по анализу поведения клиентов по их финансовым транзакциям. Твоя задача — рассуждать и объяснять, почему клиент относится к определенному полу на основе транзакций и агрегированных данных."

    user_prompt = f"""
Данные клиента:

1. Сэмпл транзакций (Markdown таблица):

Первые 50 транзакций:
{md_table_first}

Последние 50 транзакций:
{md_table_last}

2. Агрегированные показатели:
{agg_str}

3. Истинный пол клиента: {true_label}

Задача:
- Проанализировать транзакции и агрегаты.
- Рассуждать, какие признаки поведения (тип транзакций, суммы, MCC-коды, распределение расходов и т.д.) могут указывать на пол.
- Выдать подробное объяснение, почему пол такой-то.
- Объяснение должно быть логичным, опираться на данные, без догадок.
"""

# Формат ответа:
# 1. "Пол клиента: Мужчина/Женщина" - ответ должен быть только из этих двух вариантов, нельзя отвечать незнаю, не уверен и т.д.
# 2. Подробное объяснение:
# - [пункт 1]
# - [пункт 2]
# - ...
# """

    return {"system_prompt": system_prompt, "user_prompt": user_prompt}

# generate_reasoning_prompt(markdown_tables_first_100_list[0], markdown_tables_last_100_list[0], stats_descriptions_list[0], "Мужчина")

### Gen prompts without true label

In [ ]:
customers_info = [{
    "transactions_first_100": t1, 
    "transactions_last_100": t2, 
    "aggregates": agg, 
    "gender": gender,
    **generate_reasoning_prompt(t1, t2, agg, gender),
    } 
                  for t1, t2, agg, gender in tqdm(zip(markdown_tables_first_100_list, markdown_tables_last_100_list, stats_descriptions_list, y.to_list()))]

import json
with open("../data/customers_info.json", "w") as f:
    json.dump(customers_info, f, ensure_ascii=False, indent=4)

### Gen prompts with true label

In [ ]:
for i, info in enumerate(customers_info):
    prompts = generate_reasoning_prompt(info["transactions_first_100"], info["transactions_last_100"], info["aggregates"], info["gender"])

    customers_info[i]["system_prompt"] = prompts["system_prompt"]
    customers_info[i]["user_prompt"] = prompts["user_prompt"]

import json
with open("../data/customers_info_true_gender.json", "w") as f:
    json.dump(customers_info, f, ensure_ascii=False, indent=4)

### Gen prompt for both genders

In [ ]:
both_genders_test_customers = []
for i, info in tqdm(enumerate(customers_info)):
    for gender in ["мужчина", "женщина"]:
        new_customer = info.copy()
        prompts = generate_reasoning_prompt(info["transactions_first_100"], info["transactions_last_100"], info["aggregates"], gender)

        new_customer["system_prompt"] = prompts["system_prompt"]
        new_customer["user_prompt"] = prompts["user_prompt"]
        new_customer["gender"] = gender
        new_customer["customer_id"] = i

        both_genders_test_customers.append(new_customer)

import json
with open("../data/customers_info_both_genders.json", "w") as f:
    json.dump(both_genders_test_customers, f, ensure_ascii=False, indent=4)